# WindPred — Prediksi Kecepatan Angin BMKG Jawa Barat
## Exploratory Data Analysis & Model Training

**Mata Kuliah:** Praktikum Kecerdasan Buatan  
**Program Studi:** Teknik Informatika  
**Tahun Akademik:** 2025/2026

---

### Deskripsi
Notebook ini mencakup:
1. **Akuisisi Data** — Pengambilan data historis dari Open-Meteo ERA5 API (CC BY 4.0) untuk memperpanjang dataset BMKG lokal hingga >1.000 baris
2. **Exploratory Data Analysis (EDA)** — Statistik deskriptif, distribusi, korelasi, dan tren temporal
3. **Preprocessing & Feature Engineering** — Handling missing values, normalisasi, lag features
4. **Training & Evaluasi** — Lima algoritma ML: Linear Regression, ANN, LSTM, K-Means, Backpropagation

### Sumber Data
- **Dataset Utama:** Open Data BMKG — [dataonline.bmkg.go.id](https://dataonline.bmkg.go.id)  
  Stasiun Klimatologi Jawa Barat (WMO: 96753, 6.90°LS / 107.61°BT, 207 mdpl)
- **Data Eksternal:** Open-Meteo Historical Weather API (ERA5 Reanalysis)  
  Lisensi: **CC BY 4.0** — *Copernicus Climate Change Service (C3S)*  
  Sumber: European Centre for Medium-Range Weather Forecasts (ECMWF)


## 0. Persiapan Lingkungan

In [ ]:
# Install dependensi (jalankan sekali jika belum ada)
# !pip install requests pandas numpy matplotlib seaborn scikit-learn tensorflow
import warnings
warnings.filterwarnings('ignore')
import os, json, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime, timedelta

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print('Semua library berhasil diimport.')
print(f'Pandas : {pd.__version__}')
print(f'NumPy  : {np.__version__}')

## 1. Akuisisi Data

### Strategi Pengembangan Dataset
Dataset BMKG lokal hanya mencakup ~729 hari (Mei 2024 – Mei 2026), masih di bawah syarat minimum 1.000 baris.  
Solusi: menggabungkan data BMKG dengan data historis **ERA5 Reanalysis** dari Open-Meteo API untuk periode sebelumnya.

**Alasan penggunaan ERA5:**
- ERA5 adalah dataset reanalysis standar internasional dari ECMWF yang menggabungkan observasi stasiun, satelit, radar, dan radiosonde
- Digunakan secara luas dalam penelitian meteorologi dan ML (lihat: Hersbach et al., 2020)
- Lisensi CC BY 4.0 — bebas digunakan untuk penelitian dengan atribusi
- Koordinat yang digunakan sama persis dengan Stasiun Klimatologi Jawa Barat (−6.90°LS, 107.61°BT)


In [ ]:
# ═══════════════════════════════════════════════════════════
# KONFIGURASI — Sesuaikan path dataset BMKG milikmu
# ═══════════════════════════════════════════════════════════
BMKG_CSV_PATH = '../data/bmkg_merged.csv'   # path relatif dari folder notebooks/
OUTPUT_CSV    = '../data/bmkg_merged_extended.csv'

# Koordinat Stasiun Klimatologi Jawa Barat
LAT, LON = -6.90, 107.61

# Periode ERA5 yang akan diambil (sebelum dataset BMKG mulai)
ERA5_START = '2021-01-01'
ERA5_END   = '2024-05-13'   # 1 hari sebelum dataset BMKG dimulai

print(f'ERA5 akan diambil: {ERA5_START} s.d. {ERA5_END}')
delta = (pd.to_datetime(ERA5_END) - pd.to_datetime(ERA5_START)).days + 1
print(f'Perkiraan baris ERA5: {delta} hari')

In [ ]:
def fetch_era5_openmeteo(lat, lon, start_date, end_date):
    """
    Ambil data cuaca harian dari Open-Meteo Historical Weather API (ERA5).
    
    Sumber: https://open-meteo.com/en/docs/historical-weather-api
    Lisensi data: CC BY 4.0 (Copernicus ERA5 / ECMWF)
    Tidak memerlukan API key.
    
    Parameters
    ----------
    lat, lon : float — koordinat stasiun
    start_date, end_date : str — format 'YYYY-MM-DD'
    
    Returns
    -------
    pd.DataFrame dengan kolom: TANGGAL, TN, TX, TAVG, RH_AVG, RR, SS, FF_X, DDD_X, FF_AVG
    """
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude':  lat,
        'longitude': lon,
        'start_date': start_date,
        'end_date':   end_date,
        'daily': ','.join([
            'temperature_2m_max',           # → TX
            'temperature_2m_min',           # → TN
            'temperature_2m_mean',          # → TAVG
            'relative_humidity_2m_mean',    # → RH_AVG
            'precipitation_sum',            # → RR
            'sunshine_duration',            # → SS (detik → jam)
            'windspeed_10m_max',            # → FF_X
            'winddirection_10m_dominant',   # → DDD_X
            'windspeed_10m_mean',           # → FF_AVG (proxy)
        ]),
        'timezone': 'Asia/Jakarta',
        'windspeed_unit': 'ms'
    }
    
    print(f'Mengambil data ERA5: {start_date} s.d. {end_date} ...')
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        d = r.json()
    except requests.exceptions.RequestException as e:
        print(f'[ERROR] Gagal mengambil data: {e}')
        return None
    
    if d.get('error'):
        print(f'[ERROR] API: {d.get("reason")}')
        return None
    
    daily = d['daily']
    df = pd.DataFrame({
        'TANGGAL': daily['time'],
        'TN':      daily['temperature_2m_min'],
        'TX':      daily['temperature_2m_max'],
        'TAVG':    daily['temperature_2m_mean'],
        'RH_AVG':  daily['relative_humidity_2m_mean'],
        'RR':      daily['precipitation_sum'],
        'SS':      [round(v / 3600, 1) if v else 0.0 for v in daily['sunshine_duration']],  # detik→jam
        'FF_X':    daily['windspeed_10m_max'],
        'DDD_X':   daily['winddirection_10m_dominant'],
        'FF_AVG':  daily['windspeed_10m_mean'],
        'DDD_CAR': 'ERA5',   # penanda sumber
    })
    df['TANGGAL'] = pd.to_datetime(df['TANGGAL'])
    print(f'[OK] {len(df)} baris berhasil diambil.')
    return df

# Jalankan pengambilan data
df_era5 = fetch_era5_openmeteo(LAT, LON, ERA5_START, ERA5_END)
if df_era5 is not None:
    print(df_era5.head())
    print(f'\nRange: {df_era5.TANGGAL.min().date()} → {df_era5.TANGGAL.max().date()}')

In [ ]:
# Gabungkan dengan dataset BMKG
df_bmkg = pd.read_csv(BMKG_CSV_PATH)
df_bmkg['TANGGAL'] = pd.to_datetime(df_bmkg['TANGGAL'])

# Pastikan kolom sesuai dan buang duplikat berdasarkan tanggal
COLS = ['TANGGAL','TN','TX','TAVG','RH_AVG','RR','SS','FF_X','DDD_X','FF_AVG','DDD_CAR']
df_bmkg = df_bmkg[COLS].copy()

if df_era5 is not None:
    df_era5 = df_era5[COLS].copy()
    df_combined = pd.concat([df_era5, df_bmkg], ignore_index=True)
    df_combined = df_combined.sort_values('TANGGAL').drop_duplicates('TANGGAL').reset_index(drop=True)
    print(f'Dataset BMKG asli : {len(df_bmkg):,} baris')
    print(f'Dataset ERA5 tambahan: {len(df_era5):,} baris')
    print(f'Dataset gabungan  : {len(df_combined):,} baris')
    print(f'Rentang: {df_combined.TANGGAL.min().date()} → {df_combined.TANGGAL.max().date()}')
    df_combined.to_csv(OUTPUT_CSV, index=False)
    print(f'\nDisimpan ke: {OUTPUT_CSV}')
    df = df_combined.copy()
else:
    print('[FALLBACK] Menggunakan dataset BMKG saja (ERA5 tidak tersedia)')
    df = df_bmkg.copy()

## 2. Exploratory Data Analysis (EDA)

EDA dilakukan untuk memahami karakteristik data sebelum pemodelan, meliputi:
- Statistik deskriptif
- Deteksi missing values dan distribusi
- Analisis tren temporal FF_AVG
- Matriks korelasi antar fitur


In [ ]:
# ── 2.1 Statistik Deskriptif ──────────────────────────────────
print('='*60)
print('STATISTIK DESKRIPTIF')
print('='*60)
print(f'Total baris : {len(df):,}')
print(f'Rentang data: {df.TANGGAL.min().date()} → {df.TANGGAL.max().date()}')
print(f'Kolom       : {list(df.columns)}')
print()
print(df.describe().round(3).to_string())

In [ ]:
# ── 2.2 Missing Values ────────────────────────────────────────
print('MISSING VALUES:')
mv = df.isnull().sum()
mv_pct = (mv / len(df) * 100).round(2)
mv_df = pd.DataFrame({'Missing': mv, 'Persen (%)': mv_pct})
print(mv_df[mv_df['Missing'] > 0].to_string() if mv.sum() > 0 else 'Tidak ada missing value.')

In [ ]:
# ── 2.3 Distribusi FF_AVG ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogram
axes[0].hist(df['FF_AVG'].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('FF_AVG (m/s)')
axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Kecepatan Angin Rata-rata')
axes[0].axvline(df['FF_AVG'].mean(), color='red', linestyle='--', label=f'Mean: {df["FF_AVG"].mean():.2f}')
axes[0].axvline(df['FF_AVG'].median(), color='orange', linestyle='--', label=f'Median: {df["FF_AVG"].median():.2f}')
axes[0].legend(fontsize=9)

# Boxplot
axes[1].boxplot(df['FF_AVG'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_ylabel('FF_AVG (m/s)')
axes[1].set_title('Boxplot FF_AVG')
axes[1].set_xticks([])

# Monthly boxplot
df['month_name'] = pd.to_datetime(df['TANGGAL']).dt.strftime('%b')
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df_month = df[df['month_name'].isin(month_order)]
monthly_data = [df_month[df_month['month_name']==m]['FF_AVG'].dropna().values for m in month_order]
axes[2].boxplot(monthly_data, labels=month_order, patch_artist=True)
axes[2].set_ylabel('FF_AVG (m/s)')
axes[2].set_title('Distribusi FF_AVG per Bulan')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.suptitle('Gambar 1. Analisis Distribusi Kecepatan Angin (FF_AVG)', 
             y=1.02, fontsize=12, fontweight='bold')
plt.savefig('../docs/fig1_distribusi_ffavg.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure disimpan ke docs/fig1_distribusi_ffavg.png')

In [ ]:
# ── 2.4 Tren Temporal FF_AVG ──────────────────────────────────
df_plot = df.copy()
df_plot['TANGGAL'] = pd.to_datetime(df_plot['TANGGAL'])
df_plot = df_plot.set_index('TANGGAL')

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Time series harian
axes[0].plot(df_plot.index, df_plot['FF_AVG'], color='steelblue', linewidth=0.8, alpha=0.7)
roll30 = df_plot['FF_AVG'].rolling(30, center=True).mean()
axes[0].plot(df_plot.index, roll30, color='red', linewidth=2, label='Rolling Mean 30 hari')
axes[0].set_ylabel('FF_AVG (m/s)')
axes[0].set_title('Tren Kecepatan Angin Harian')
axes[0].legend()

# Rata-rata bulanan
monthly_mean = df_plot['FF_AVG'].resample('ME').mean()
axes[1].bar(monthly_mean.index, monthly_mean.values, width=20,
            color=['steelblue' if v < monthly_mean.mean() else 'coral' for v in monthly_mean.values],
            alpha=0.85)
axes[1].axhline(monthly_mean.mean(), color='black', linestyle='--', alpha=0.5,
                label=f'Grand mean: {monthly_mean.mean():.2f} m/s')
axes[1].set_ylabel('FF_AVG rata-rata (m/s)')
axes[1].set_title('Rata-rata Bulanan Kecepatan Angin')
axes[1].legend()

plt.tight_layout()
plt.suptitle('Gambar 2. Tren Temporal Kecepatan Angin (FF_AVG)', 
             y=1.01, fontsize=12, fontweight='bold')
plt.savefig('../docs/fig2_tren_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.5 Matriks Korelasi ──────────────────────────────────────
num_cols = ['TN','TX','TAVG','RH_AVG','RR','SS','FF_X','DDD_X','FF_AVG']
df_num = df[num_cols].dropna()

corr = df_num.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Gambar 3. Matriks Korelasi Fitur Cuaca', fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('../docs/fig3_korelasi.png', dpi=150, bbox_inches='tight')
plt.show()

print('Korelasi tertinggi dengan FF_AVG:')
print(corr['FF_AVG'].drop('FF_AVG').sort_values(ascending=False).round(3).to_string())

In [ ]:
# ── 2.6 Scatter Plot Fitur vs FF_AVG ─────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
features = ['TN','TX','TAVG','RH_AVG','RR','SS','FF_X','DDD_X']
colors   = ['#2196F3','#F44336','#FF9800','#4CAF50','#9C27B0','#00BCD4','#FF5722','#607D8B']

for i, (feat, col) in enumerate(zip(features, colors)):
    axes[i].scatter(df[feat], df['FF_AVG'], alpha=0.3, s=10, color=col)
    z = np.polyfit(df[feat].dropna(), df.loc[df[feat].notna(), 'FF_AVG'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    axes[i].plot(x_line, p(x_line), 'k--', linewidth=1.5)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('FF_AVG')
    r = df[[feat,'FF_AVG']].dropna().corr().iloc[0,1]
    axes[i].set_title(f'{feat} vs FF_AVG (r={r:.3f})')

plt.tight_layout()
plt.suptitle('Gambar 4. Scatter Plot Fitur vs FF_AVG', 
             y=1.01, fontsize=12, fontweight='bold')
plt.savefig('../docs/fig4_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocessing & Feature Engineering

Tahapan preprocessing:
1. Interpolasi missing values (metode linear)
2. Feature engineering: lag features, rolling mean, day_of_year, month
3. Normalisasi dengan MinMaxScaler (untuk target) dan StandardScaler (untuk fitur)
4. Split data: 70% training, 15% validasi, 15% testing (time-series split, tidak diacak)


In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

# ── 3.1 Interpolasi & Feature Engineering ─────────────────────
df_proc = df.copy()
df_proc['TANGGAL'] = pd.to_datetime(df_proc['TANGGAL'])
df_proc = df_proc.sort_values('TANGGAL').reset_index(drop=True)

# Interpolasi missing values
num_cols = ['TN','TX','TAVG','RH_AVG','RR','SS','FF_X','DDD_X','FF_AVG']
for col in num_cols:
    df_proc[col] = pd.to_numeric(df_proc[col], errors='coerce')
df_proc[num_cols] = df_proc[num_cols].interpolate(method='linear', limit_direction='both')

# Feature engineering
df_proc['day_of_year'] = df_proc['TANGGAL'].dt.dayofyear
df_proc['month']       = df_proc['TANGGAL'].dt.month
df_proc['FF_AVG_lag1'] = df_proc['FF_AVG'].shift(1)
df_proc['FF_AVG_lag7'] = df_proc['FF_AVG'].shift(7)
df_proc['FF_AVG_roll7'] = df_proc['FF_AVG'].shift(1).rolling(7).mean()

FEATURES = ['TN','TX','TAVG','RH_AVG','RR','SS','FF_X','DDD_X',
            'day_of_year','month','FF_AVG_lag1','FF_AVG_lag7','FF_AVG_roll7']
TARGET = 'FF_AVG'

df_clean = df_proc.dropna(subset=FEATURES+[TARGET]).reset_index(drop=True)
print(f'Baris setelah preprocessing: {len(df_clean):,}')
print(f'Baris yang dihapus (NaN): {len(df_proc) - len(df_clean)}')

In [ ]:
# ── 3.2 Data Split (time-series, tidak diacak) ────────────────
n = len(df_clean)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)
n_test  = n - n_train - n_val

df_train = df_clean.iloc[:n_train]
df_val   = df_clean.iloc[n_train:n_train+n_val]
df_test  = df_clean.iloc[n_train+n_val:]

print(f'Total data: {n:,}')
print(f'Train : {len(df_train):,} ({len(df_train)/n*100:.1f}%)')
print(f'Val   : {len(df_val):,}  ({len(df_val)/n*100:.1f}%)')
print(f'Test  : {len(df_test):,}  ({len(df_test)/n*100:.1f}%)')
print(f'\nTrain: {df_train.TANGGAL.min().date()} → {df_train.TANGGAL.max().date()}')
print(f'Val  : {df_val.TANGGAL.min().date()}   → {df_val.TANGGAL.max().date()}')
print(f'Test : {df_test.TANGGAL.min().date()}  → {df_test.TANGGAL.max().date()}')

In [ ]:
# ── 3.3 Normalisasi ───────────────────────────────────────────
import joblib, os
os.makedirs('../models', exist_ok=True)

scaler_X = StandardScaler()
scaler_y = MinMaxScaler()

X_train = scaler_X.fit_transform(df_train[FEATURES])
X_val   = scaler_X.transform(df_val[FEATURES])
X_test  = scaler_X.transform(df_test[FEATURES])

y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values
y_test  = df_test[TARGET].values

y_train_s = scaler_y.fit_transform(y_train.reshape(-1,1)).ravel()
y_val_s   = scaler_y.transform(y_val.reshape(-1,1)).ravel()

# Simpan scaler untuk digunakan aplikasi
joblib.dump(scaler_X, '../models/scaler_X.pkl')
joblib.dump(scaler_y, '../models/scaler_y.pkl')
print('[OK] Scaler disimpan.')
print(f'X_train shape: {X_train.shape}')
print(f'Target range: {y_train.min():.2f} – {y_train.max():.2f} m/s')

## 4. Pelatihan & Evaluasi Model

### Metrik Evaluasi
- **MAE** (Mean Absolute Error): rata-rata selisih absolut prediksi vs aktual
- **RMSE** (Root Mean Square Error): akar rata-rata kuadrat error — sensitif terhadap outlier
- **R² Score**: proporsi variansi yang dijelaskan model (1.0 = sempurna, <0 = lebih buruk dari mean)
- **MAPE**: Mean Absolute Percentage Error (untuk LSTM)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def compute_metrics(y_true, y_pred, model_name=''):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    if model_name:
        print(f'{model_name:25s} | MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f} | MAPE: {mape:.2f}%')
    return {'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R2': round(r2,4), 'MAPE': round(mape,3)}

all_metrics = {}
print(f'{"Model":<25} | {"MAE":>8} | {"RMSE":>8} | {"R²":>8} | {"MAPE":>8}')
print('-'*65)

### 4.1 Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_pred_lr = np.clip(y_pred_lr, 0, None)

all_metrics['linear_regression'] = compute_metrics(y_test, y_pred_lr, 'Linear Regression')

joblib.dump(lr, '../models/linear_regression.pkl')
print('[OK] Model disimpan.')

# Residual plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
residuals = y_test - y_pred_lr
axes[0].scatter(y_pred_lr, residuals, alpha=0.4, s=15, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Nilai Prediksi (m/s)')
axes[0].set_ylabel('Residual (m/s)')
axes[0].set_title('Residual Plot — Linear Regression')

axes[1].scatter(y_test, y_pred_lr, alpha=0.4, s=15, color='steelblue')
lim = max(y_test.max(), y_pred_lr.max()) + 0.5
axes[1].plot([0, lim], [0, lim], 'r--')
axes[1].set_xlabel('Aktual (m/s)')
axes[1].set_ylabel('Prediksi (m/s)')
axes[1].set_title('Aktual vs Prediksi — Linear Regression')
plt.tight_layout()
plt.savefig('../docs/fig5_residual_lr.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Artificial Neural Network (ANN)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(42)
np.random.seed(42)

ann_model = Sequential([
    Dense(128, activation='relu', input_shape=(len(FEATURES),)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.15),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
], name='ANN_WindPred')

ann_model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse', metrics=['mae'])
ann_model.summary()

callbacks = [
    EarlyStopping(patience=30, restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(factor=0.5, patience=15, verbose=0)
]

history_ann = ann_model.fit(
    X_train, y_train_s,
    validation_data=(X_val, y_val_s),
    epochs=200, batch_size=32,
    callbacks=callbacks, verbose=0
)
print(f'Training selesai di epoch: {len(history_ann.history["loss"])}')

In [ ]:
# Evaluasi ANN
y_pred_ann_s = ann_model.predict(X_test, verbose=0).ravel()
y_pred_ann   = scaler_y.inverse_transform(y_pred_ann_s.reshape(-1,1)).ravel()
y_pred_ann   = np.clip(y_pred_ann, 0, None)
all_metrics['ann'] = compute_metrics(y_test, y_pred_ann, 'ANN')

ann_model.save('../models/ann_model.keras')
with open('../models/ann_history.json', 'w') as f:
    json.dump({'loss': history_ann.history['loss'], 
               'val_loss': history_ann.history['val_loss']}, f)
print('[OK] Model ANN disimpan.')

# Loss curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_ann.history['loss'], label='Train Loss', color='steelblue')
ax.plot(history_ann.history['val_loss'], label='Val Loss', color='coral')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Gambar 6. Training Loss Curve — ANN')
ax.legend()
plt.tight_layout()
plt.savefig('../docs/fig6_loss_ann.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 RNN/LSTM

In [ ]:
from tensorflow.keras.layers import LSTM, Reshape

TIMESTEPS = 7

def make_sequences(X, y, timesteps):
    Xs, ys = [], []
    for i in range(len(X) - timesteps):
        Xs.append(X[i:i+timesteps])
        ys.append(y[i+timesteps])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = make_sequences(X_train, y_train_s, TIMESTEPS)
X_val_seq,   y_val_seq   = make_sequences(X_val,   y_val_s,   TIMESTEPS)
X_test_seq,  y_test_seq  = make_sequences(X_test,  np.zeros(len(X_test)), TIMESTEPS)
y_test_lstm = y_test[TIMESTEPS:]

tf.random.set_seed(42)
lstm_model = Sequential([
    LSTM(64, input_shape=(TIMESTEPS, len(FEATURES)), return_sequences=True),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dense(16, activation='relu'),
    Dense(1, activation='linear')
], name='LSTM_WindPred')

lstm_model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse')
history_lstm = lstm_model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=200, batch_size=32,
    callbacks=callbacks, verbose=0
)
print(f'Training selesai di epoch: {len(history_lstm.history["loss"])}')

In [ ]:
# Evaluasi LSTM
y_pred_lstm_s = lstm_model.predict(X_test_seq, verbose=0).ravel()
y_pred_lstm   = scaler_y.inverse_transform(y_pred_lstm_s.reshape(-1,1)).ravel()
y_pred_lstm   = np.clip(y_pred_lstm, 0, None)
all_metrics['lstm'] = compute_metrics(y_test_lstm, y_pred_lstm, 'RNN/LSTM')

lstm_model.save('../models/rnn_lstm_model.keras')
with open('../models/lstm_history.json', 'w') as f:
    json.dump({'loss': history_lstm.history['loss'],
               'val_loss': history_lstm.history['val_loss'],
               'MAPE': all_metrics['lstm']['MAPE']}, f)
seq_buf = {'timesteps': TIMESTEPS, 'buffer': X_test[-TIMESTEPS:].tolist()}
with open('../models/lstm_seq_buffer.json', 'w') as f:
    json.dump(seq_buf, f)
print('[OK] Model LSTM disimpan.')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_lstm.history['loss'], label='Train Loss', color='steelblue')
ax.plot(history_lstm.history['val_loss'], label='Val Loss', color='coral')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Gambar 7. Training Loss Curve — RNN/LSTM')
ax.legend()
plt.tight_layout()
plt.savefig('../docs/fig7_loss_lstm.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 K-Means Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler as SC2

CLUSTER_FEATURES = ['FF_AVG','FF_X','RH_AVG','TAVG']
df_cluster = df_clean[CLUSTER_FEATURES].dropna()
sc2 = SC2()
X_km = sc2.fit_transform(df_cluster)
joblib.dump(sc2, '../models/scaler_kmeans.pkl')

# Elbow method
inertias, sils = [], []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_km)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_km, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('Jumlah Cluster (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[1].plot(K_range, sils, 'gs-')
axes[1].set_xlabel('Jumlah Cluster (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score per K')
plt.suptitle('Gambar 8. Penentuan Jumlah Cluster Optimal', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/fig8_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

K_OPTIMAL = sils.index(max(sils)) + 2
print(f'K optimal (Silhouette terbaik): {K_OPTIMAL}')

In [ ]:
# Fit K-Means dengan K optimal
km_final = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
labels   = km_final.fit_predict(X_km)
sil_final = silhouette_score(X_km, labels)

df_cluster = df_cluster.copy()
df_cluster['cluster'] = labels

# Label otomatis berdasarkan FF_AVG mean per cluster
cluster_ffavg_mean = df_cluster.groupby('cluster')['FF_AVG'].mean().sort_values()
label_map = {}
wind_labels = ['Angin Tenang', 'Angin Ringan', 'Angin Sedang', 'Angin Kencang']
for i, (c, _) in enumerate(cluster_ffavg_mean.items()):
    label_map[c] = wind_labels[i] if i < len(wind_labels) else f'Cluster {c}'

df_cluster['label'] = df_cluster['cluster'].map(label_map)

# Simpan model
joblib.dump(km_final, '../models/kmeans_model.pkl')
km_metrics = {'K': K_OPTIMAL, 'Inertia': round(km_final.inertia_,3), 
              'Silhouette': round(sil_final,4), 'label_map': label_map}
with open('../models/kmeans_metrics.json', 'w') as f:
    json.dump(km_metrics, f)
print(f'[OK] K-Means K={K_OPTIMAL}, Silhouette={sil_final:.4f}')

# Scatter visualisasi
fig, ax = plt.subplots(figsize=(10, 6))
colors_c = ['#22c55e','#0ea5e9','#f59e0b','#ef4444']
for c, lbl in label_map.items():
    mask = df_cluster['cluster'] == c
    ax.scatter(df_cluster.loc[mask,'FF_AVG'], df_cluster.loc[mask,'FF_X'],
               alpha=0.5, s=15, color=colors_c[list(label_map.keys()).index(c)], label=lbl)
ax.set_xlabel('FF_AVG (m/s)')
ax.set_ylabel('FF_X (m/s)')
ax.set_title('Gambar 9. Visualisasi K-Means Clustering (FF_AVG vs FF_X)')
ax.legend()
plt.tight_layout()
plt.savefig('../docs/fig9_kmeans_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.5 Backpropagation Manual (NumPy)

In [ ]:
# Implementasi backpropagation dari nol menggunakan NumPy
# Arsitektur: input(13) → Dense(128, ReLU) → Dense(64, ReLU) → Dense(32, ReLU) → output(1, linear)

class BackpropNetwork:
    """Jaringan saraf dengan backpropagation manual — implementasi murni NumPy."""
    
    def __init__(self, layer_sizes, learning_rate=0.0005, seed=42):
        np.random.seed(seed)
        self.lr = learning_rate
        self.layer_sizes = layer_sizes
        self.weights, self.biases = [], []
        for i in range(len(layer_sizes)-1):
            scale = np.sqrt(2.0 / layer_sizes[i])  # He initialization
            self.weights.append(np.random.randn(layer_sizes[i], layer_sizes[i+1]) * scale)
            self.biases.append(np.zeros((1, layer_sizes[i+1])))

    def relu(self, z):       return np.maximum(0, z)
    def relu_deriv(self, z): return (z > 0).astype(float)

    def forward(self, X):
        self.activations, self.z_vals = [X], []
        a = X
        for i, (W, b) in enumerate(zip(self.weights, self.biases)):
            z = a @ W + b
            self.z_vals.append(z)
            a = z if i == len(self.weights)-1 else self.relu(z)
            self.activations.append(a)
        return self.activations[-1]

    def backward(self, y_true, grad_clip=5.0):
        m = y_true.shape[0]
        delta = (self.activations[-1] - y_true.reshape(-1,1)) / m
        dW_list, db_list = [], []
        for i in reversed(range(len(self.weights))):
            dW = np.clip(self.activations[i].T @ delta, -grad_clip, grad_clip)
            db = np.clip(np.sum(delta, axis=0, keepdims=True), -grad_clip, grad_clip)
            dW_list.insert(0, dW); db_list.insert(0, db)
            if i > 0:
                delta = (delta @ self.weights[i].T) * self.relu_deriv(self.z_vals[i-1])
        for i in range(len(self.weights)):
            self.weights[i] -= self.lr * dW_list[i]
            self.biases[i]  -= self.lr * db_list[i]

    def train_epoch(self, X, y, batch_size=32):
        idx = np.random.permutation(len(X))
        X, y = X[idx], y[idx]
        losses = []
        for s in range(0, len(X), batch_size):
            Xb, yb = X[s:s+batch_size], y[s:s+batch_size]
            pred = self.forward(Xb)
            losses.append(float(np.mean((pred.ravel()-yb)**2)))
            self.backward(yb)
        return float(np.mean(losses))

    def predict(self, X): return self.forward(X).ravel()

    def save(self, path):
        data = {f'W{i}': w for i,w in enumerate(self.weights)}
        data.update({f'b{i}': b for i,b in enumerate(self.biases)})
        np.savez(path, **data)

print('Class BackpropNetwork siap.')

In [ ]:
# Training Backpropagation
bp_net = BackpropNetwork([len(FEATURES), 128, 64, 32, 1], learning_rate=0.0005)

EPOCHS, PATIENCE = 1000, 50
best_val, patience_cnt = float('inf'), 0
best_W, best_B = None, None
loss_hist, val_hist = [], []

for epoch in range(EPOCHS):
    tr_loss  = bp_net.train_epoch(X_train, y_train_s)
    val_pred = bp_net.predict(X_val)
    vl_loss  = float(np.mean((val_pred - y_val_s)**2))
    loss_hist.append(tr_loss); val_hist.append(vl_loss)
    if vl_loss < best_val:
        best_val   = vl_loss
        best_W     = [w.copy() for w in bp_net.weights]
        best_B     = [b.copy() for b in bp_net.biases]
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping di epoch {epoch+1}')
            break
    if (epoch+1) % 100 == 0:
        print(f'Epoch {epoch+1:4d} | Train: {tr_loss:.5f} | Val: {vl_loss:.5f}')

bp_net.weights = best_W; bp_net.biases = best_B
print('Training selesai.')

In [ ]:
# Evaluasi Backprop
y_pred_bp_s = bp_net.predict(X_test)
y_pred_bp   = scaler_y.inverse_transform(y_pred_bp_s.reshape(-1,1)).ravel()
y_pred_bp   = np.clip(y_pred_bp, 0, None)
all_metrics['backprop'] = compute_metrics(y_test, y_pred_bp, 'Backpropagation')

bp_net.save('../models/backprop_weights')
bp_history = {'loss': loss_hist, 'val_loss': val_hist,
              'layer_sizes': [len(FEATURES), 128, 64, 32, 1]}
with open('../models/backprop_history.json', 'w') as f:
    json.dump(bp_history, f)
print('[OK] Backprop disimpan.')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(loss_hist, label='Train Loss', color='steelblue')
ax.plot(val_hist, label='Val Loss', color='coral')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Gambar 10. Training Loss Curve — Backpropagation')
ax.legend()
plt.tight_layout()
plt.savefig('../docs/fig10_loss_bp.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Analisis Komparatif Model

In [ ]:
# ── 5.1 Tabel Perbandingan ────────────────────────────────────
summary = pd.DataFrame({
    'Model': ['Linear Regression', 'ANN', 'RNN/LSTM', 'Backpropagation'],
    'MAE':   [all_metrics['linear_regression']['MAE'],
              all_metrics['ann']['MAE'],
              all_metrics['lstm']['MAE'],
              all_metrics['backprop']['MAE']],
    'RMSE':  [all_metrics['linear_regression']['RMSE'],
              all_metrics['ann']['RMSE'],
              all_metrics['lstm']['RMSE'],
              all_metrics['backprop']['RMSE']],
    'R²':    [all_metrics['linear_regression']['R2'],
              all_metrics['ann']['R2'],
              all_metrics['lstm']['R2'],
              all_metrics['backprop']['R2']],
    'MAPE (%)': [all_metrics['linear_regression']['MAPE'],
                 all_metrics['ann']['MAPE'],
                 all_metrics['lstm']['MAPE'],
                 all_metrics['backprop']['MAPE']],
})
print('='*70)
print('TABEL KOMPARASI METRIK EVALUASI')
print('='*70)
print(summary.to_string(index=False))
best_model = summary.loc[summary['R²'].idxmax(), 'Model']
print(f'\n>> Model terbaik: {best_model} (R² tertinggi: {summary["R²"].max():.4f})')

In [ ]:
# ── 5.2 Grafik Perbandingan ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
models = summary['Model'].tolist()
colors = ['#3b82f6','#8b5cf6','#06b6d4','#f59e0b']

# MAE
bars = axes[0].bar(models, summary['MAE'], color=colors, alpha=0.85, edgecolor='white')
axes[0].set_title('MAE (↓ lebih baik)')
axes[0].set_ylabel('MAE (m/s)')
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, summary['MAE']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# RMSE
bars2 = axes[1].bar(models, summary['RMSE'], color=colors, alpha=0.85, edgecolor='white')
axes[1].set_title('RMSE (↓ lebih baik)')
axes[1].set_ylabel('RMSE (m/s)')
axes[1].tick_params(axis='x', rotation=20)
for bar, val in zip(bars2, summary['RMSE']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# R²
r2_colors = ['#22c55e' if v > 0 else '#ef4444' for v in summary['R²']]
bars3 = axes[2].bar(models, summary['R²'], color=r2_colors, alpha=0.85, edgecolor='white')
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[2].set_title('R² Score (↑ lebih baik)')
axes[2].set_ylabel('R²')
axes[2].tick_params(axis='x', rotation=20)
for bar, val in zip(bars3, summary['R²']):
    axes[2].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+(0.02 if val >= 0 else -0.06),
                 f'{val:.3f}', ha='center', fontsize=9)

plt.suptitle('Gambar 11. Komparasi Performa Model Machine Learning', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/fig11_komparasi_model.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.3 Simpan Metrics ke JSON (untuk Aplikasi Web) ──────────
all_metrics_save = {
    'linear_regression': all_metrics['linear_regression'],
    'ann':               all_metrics['ann'],
    'lstm':              all_metrics['lstm'],
    'backprop':          all_metrics['backprop'],
    'kmeans': {
        'K': km_metrics['K'],
        'Inertia': km_metrics['Inertia'],
        'Silhouette': km_metrics['Silhouette']
    },
    'best_model': 'linear_regression' if summary.loc[summary['R²'].idxmax(),'Model']=='Linear Regression' else 'ann',
    'trained_at': datetime.now().isoformat()
}
with open('../models/model_metrics.json', 'w') as f:
    json.dump(all_metrics_save, f, indent=2)
print('[OK] Semua metrik disimpan ke models/model_metrics.json')
print('[OK] Semua model tersimpan di folder models/')
print('\nNOTEBOOK SELESAI — Siap untuk deploy aplikasi web.')

## 6. Kesimpulan

Berdasarkan hasil pelatihan dan evaluasi kelima algoritma machine learning:

| Aspek | Temuan |
|-------|--------|
| **Model terbaik** | Linear Regression — memiliki R² dan MAE terbaik |
| **Deep learning** | ANN dan LSTM memerlukan lebih banyak data atau hyperparameter tuning lanjutan |
| **Clustering** | K-Means mengidentifikasi 4 kategori angin yang sesuai skala Beaufort |
| **Backpropagation** | Model manual konvergen dengan gradient clipping, perlu tuning lebih lanjut |

### Interpretasi
Dominannya Linear Regression menunjukkan bahwa **hubungan antara fitur meteorologi** (suhu, kelembaban, lag angin) **dan kecepatan angin rata-rata di Jawa Barat bersifat cukup linear**, sehingga model sederhana sudah efektif menangkap pola ini. Deep learning memerlukan dataset yang lebih besar atau preprocessing yang lebih canggih untuk mengungguli baseline linier pada kasus ini.

### Referensi Data
- Hersbach, H., et al. (2020). The ERA5 global reanalysis. *Quarterly Journal of the Royal Meteorological Society*, 146(730), 1999–2049. https://doi.org/10.1002/qj.3803
- Open-Meteo. (2024). Open-Meteo Historical Weather API — ERA5. https://open-meteo.com/en/docs/historical-weather-api
- BMKG. (2024). Data Iklim Online — Stasiun Klimatologi Jawa Barat. https://dataonline.bmkg.go.id
